# Mosaic AI Agent Framework: Author and deploy a single turn multi-Genie agent with DSPy
This notebook demonstrates how to build an agent system using Mosaic AI Agent Framework and [DSPy](https://dspy.ai/), where [Genie](https://www.databricks.com/product/business-intelligence/ai-bi-genie) is one of the agents. In this notebook, you will:

1. Author an agent system using DSPy.
1. Wrap the DSPy agent with MLflow ChatAgent to ensure compatibility with Databricks features.
1. Manually test the agent system's output.
1. Log and deploy the agent system.

## Why use a Genie agent?
Agentic systems consist of multiple tools and AI agents working together, each with specialized capabilities. 

Unlike SQL functions which can only run pre-defined queries, Genie has the flexibility to create novel queries to answer user questions.

Due to this, Genie becomes a core component of multi-agent solutions which require answering questions over structured data.

## Prerequisites 

1. Address all TODOs in this notebook.

## Notebook contents

1. **Create Genie Spaces** - Create two Genie Spaces for use with our agent (See Databricks documentation for more details ([AWS](https://docs.databricks.com/aws/en/genie/set-up) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/genie/set-up))). This notebook includes steps to obtain data to use with the Genie data rooms, followed by steps for creating the Genie rooms themselves. You can choose to use your own data if you wish.
1. **Write your agent using DSPy**
1. **Test the agent**
1. **Log the agent as an MLflow model**
1. **Register the model in Unity Catalog**
1. **Deploy the agent**

### Install dependencies

In [0]:
%pip install -qqqq --upgrade dspy mlflow databricks-sdk databricks-agents "pydantic>=2.0.0,<3.0.0" uv databricks-ai-bridge
dbutils.library.restartPython()

## 1. Create Genie Spaces

### Load Marketplace data for Genie Rooms
Skip this section if you have existing data you want to use

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

In [0]:
from databricks.sdk.service.marketplace import ConsumerTerms

def get_current_user()-> str:
  return (dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .userName()
    .get()
    .split("@")[0]
    .replace(".", "_"))

def get_terms_of_service_version(terms_of_service: str)->str:
  return terms_of_service.split("/files/")[1].split("/")[0]

def install_marketplace_tables(listing, catalog_suffix:str) -> None:
  consumer_terms_version = get_terms_of_service_version(listing.detail.terms_of_service)
  w.consumer_installations.create(
    listing_id=listing.id, 
    share_name=listing.summary.share.name, 
    catalog_name=f"{get_current_user()}_{catalog_suffix}",
    accepted_consumer_terms=ConsumerTerms(version=consumer_terms_version)
  )

def delete_marketplace_tables(listing_id:str, catalog_suffix:str) -> None:
  installation = [
      installation
      for installation in w.consumer_installations.list()
      if installation.catalog_name == f"{get_current_user()}_{catalog_suffix}"
  ][0]

  w.consumer_installations.delete(
      installation_id=installation.id,
      listing_id=listing_id,
  )

### Load Retail Catalog from Marketplace

In [0]:
simulated_retail_listing = [marketplace_list for marketplace_list in w.consumer_listings.list() if marketplace_list.summary.name=='Simulated Retail Customer Data'][0]

install_marketplace_tables(listing=simulated_retail_listing, catalog_suffix="us_retail")

### Load Australia Sales Catalog from Marketplace

In [0]:
australia_sales_listing = [marketplace_list for marketplace_list in w.consumer_listings.list() if marketplace_list.summary.name=='Simulated Australia Sales and Opportunities Data'][0]

install_marketplace_tables(listing=australia_sales_listing, catalog_suffix="australia_sales")

### Create 2 Genie Data Rooms: 

After running the cells above, you will have the following delta shared catalogs available in your workspace: 
1. `<your_username>_us_retail` - contains US retail data
1. `<your_username>_australia_sales` - contains Australian sales data

**TODO:** Follow the steps given in the official docs ([AWS](https://docs.databricks.com/aws/en/genie/set-up) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/genie/set-up)) to create 2 Genie data rooms:
1. Which contains all the tables in the `{your_username}_us_retail.v01` schema
1. Which contains all the tables in the `{your_username}_australia_sales.v01` schema  

## 2. Write your agent using DSPy

The key parts of our DSPy agent will be:

1. An LLM that we can configure via dspy.configure
2. A dspy.Signature, that defines how the LLM should accomplish the task. A signature structures your inputs and outputs, enforces typing and provides additional instructions. 
3. A dspy.Module to convert the signature into a prompt. Depending on the module, you can enable agents (dspy.ReAct), chain of thought or reasoning (dspy.CoT) or just a simple predict (dspy.Predict). The module's output will be a prediction where we can programatically access the outputs.

To learn more about DSPy, check out the [DSPy docs](https://dspy.ai/).

**TODO**: Fill in the values below and run the cell to generate a config.yaml file. Your agent will access the values from this file. 

You can find your Genie space ID in the URL of each respective Genie space. URLs are of the format: `https://<databricks-instance>/genie/rooms/<space-id>`

In [0]:
%%writefile config.yaml

# You can specify a different llm endpoint if you wish to
llm_endpoint_name: "databricks-meta-llama-3-3-70b-instruct"

australian_sales_genie_space_id: "<your_australia_genie_space_id>"
us_retail_genie_space_id: "<your_us_genie_space_id>"

sql_warehouse_id: "<your_sql_warehouse_id>"

# For registering your mlflow model
catalog: "<your_catalog>"
schema: "<your_schema>"
model_name: "<your_model_name>"

In [0]:
import mlflow

# Get the notebook experiment id
experiment_id = mlflow.tracking.fluent._get_experiment_id()
print(experiment_id)

 Now, use `%%writefile` to create a Python file containing the agent's code.

In [0]:
%%writefile agent.py

import mlflow
from mlflow.models import ModelConfig
from typing import Any, Generator, Optional
from databricks.sdk import WorkspaceClient
from mlflow.entities import SpanType

from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
)

from pydantic import BaseModel
from databricks_ai_bridge.genie import Genie

import dspy
import uuid
import datetime
import os
import json
import re
from uuid import NAMESPACE_DNS, uuid3, uuid4

# Autolog DSPy traces to MLflow
mlflow.dspy.autolog()

config_file = "config.yaml"
model_config = mlflow.models.ModelConfig(development_config=config_file)

# Set up DSPy with a Databricks-hosted LLM
LLM_ENDPOINT_NAME = model_config.get("llm_endpoint_name")
lm = dspy.LM(model=f"databricks/{LLM_ENDPOINT_NAME}")
dspy.settings.configure(lm=lm)


######################################
## Create our Genie Selector Signature
######################################
# This signature is used to determine which Genie Space to use based on the request, and then it uses self-reasoning to select the proper tool to use with the relevant Genie Space.

# Deciding on your module's inputs, outputs, and their types is the most critical part of creating a signature. This definition forms the contract that other components in your multi-agent pipeline will rely on.

class genie_selector_agent(dspy.Signature):
  """
  Given a question, determine which genie space tool to call, send the exact question to the tool and answer the question given the response from the tool.
  """ 
  # **NOTE** : You can add details to the prompt about each tool to increase the accuracy of the agent. Both the function name and prompt inform the agent what the tool is used for.

  question: str = dspy.InputField()
  response: str = dspy.OutputField() 
  sql_query_output:  list = dspy.OutputField()


# Create custom tool calling message for streaming purposes.
class MyStatusMessageProvider(dspy.streaming.StatusMessageProvider):
    def tool_start_status_message(self, instance, inputs):
        tool_calling_info = {
            "tool_name": instance.name,
            "tool_args": inputs["kwargs"],
        }
        return json.dumps(tool_calling_info)

    def tool_end_status_message(self, outputs):
        tool_result = {"tool_result": outputs}
        return json.dumps(tool_result)


#####################################
## Create our DSPY Chat Agent Class
#####################################
class DSPyResponsesAgent(ResponsesAgent):     
    def __init__(self):
      self.w = WorkspaceClient(
        host=os.getenv("DATABRICKS_HOST"),
        token=os.getenv("DATABRICKS_TOKEN")
      )

      self.lm = lm
      self.genie_selector_agent_sig = genie_selector_agent

      # Create the genie space functions dynamically
      self.australian_sales_tool = self._create_genie_space_tool(
          genie_space_id=model_config.get("australian_sales_genie_space_id"),
          name="australian_sales_genie_tool",
          description="This genie space is used to query data about Australia Sales. This genie space takes in a request and returns an answer based on the relevant data."
      )
      
      self.us_sales_tool = self._create_genie_space_tool(
          genie_space_id=model_config.get("us_retail_genie_space_id"),
          name="us_sales_genie_tool",
          description="This genie space is used to query data about USA Retail Sales. This genie space takes in a request and returns an answer based on the relevant data."
      )

      self.multi_genie_agent = dspy.ReAct(self.genie_selector_agent_sig, 
                                          tools=[self.australian_sales_tool, self.us_sales_tool], 
                                          max_iters=1)
      
      # Convert the agent to be streaming-compatible.
      self._streamified_agent = dspy.streamify(
          self.multi_genie_agent,
          status_message_provider=MyStatusMessageProvider(),
          stream_listeners=[
              dspy.streaming.StreamListener(signature_field_name="next_thought", allow_reuse=True),
              dspy.streaming.StreamListener(signature_field_name="response"),
              dspy.streaming.StreamListener(signature_field_name="reasoning"),
          ],
          async_streaming=False,
      )
      
      # Agent internal states for streaming
      self._concated_stream_chunks = [[]]
      self._last_tool_call_id = None


    #######################################################
    ## Generic Function Creator for Genie Spaces
    #######################################################
    def _create_genie_space_tool(self, genie_space_id: str, name: str, description: str):
      """Creates a dspy Tool that queries a specific Genie space using databricks-ai-bridge.
      
      Args:
          genie_space_id: the key of the genie space ID
          name: tool name - used by ReAct to select the correct tool
          description: description of the tool - used by ReAct to select the correct tool
      
      Returns:
          A dspy Tool that queries the specified Genie space
      """
      # Create a Genie instance for this space
      genie = Genie(space_id=genie_space_id, client=self.w, return_pandas=False)
      
      def genie_space_query(question):
        """Query the Genie space and return results."""
        # Use the ask_question method which handles conversation management and polling
        genie_response = genie.ask_question(f"{question} always limit to one result")
        
        # Return the result from the GenieResponse object
        return genie_response.result
    
      return dspy.Tool(genie_space_query, name=name, desc=description) 


    def _dspy_stream_chunk_to_responses(self, chunk) -> dict[str, Any]:
        """Convert from DSPy streaming chunks to Responses output item dictionaries"""

        if isinstance(chunk, dspy.streaming.StatusMessage):
            # Extract tool calling information to form the clean streaming chunks.
            message_dict = json.loads(chunk.message)
            if "tool_name" in message_dict:
                # Set a new tool call ID when detecting a new tool call.
                self._last_tool_call_id = str(uuid4())
                return self.create_function_call_item(
                    id=str(uuid4()),
                    call_id=self._last_tool_call_id,
                    name=message_dict["tool_name"],
                    arguments=json.dumps(message_dict["tool_args"]),
                )
            elif "tool_result" in message_dict:
                # `call_id` in the result chunk has to match the `call_id` from tool invocation chunk.
                call_id = self._last_tool_call_id
                self._last_tool_call_id = None
                return self.create_function_call_output_item(
                    call_id=call_id,
                    output=json.dumps(message_dict["tool_result"]),
                )
        elif isinstance(chunk, dspy.streaming.StreamResponse):
            stream_chunk = self.create_text_delta(
                delta=chunk.chunk,
                item_id=str(
                    uuid3(
                        NAMESPACE_DNS,
                        f"{chunk.predict_name}.{chunk.signature_field_name}.stream_count{len(self._concated_stream_chunks)}",
                    )
                ),  # Generate a deterministic ID because streaming chunks from the same LM call should be grouped together.
            )
            self._concated_stream_chunks[-1].append(chunk.chunk)
            return stream_chunk

        elif isinstance(chunk, dspy.Prediction):
            # `dspy.Prediction` is the last stream chunk that contains the final answer.
            return self.create_text_output_item(text=chunk.response, id=str(uuid4()))


    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [event.item for event in self.predict_stream(request) if event.type == "response.output_item.done"]
        return ResponsesAgentResponse(output=[outputs[-1]])

    
    def predict_stream(
        self, 
        request: ResponsesAgentRequest
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        last_message = request.input[-1]
        if last_message.role != "user":
            raise ValueError("No user question detected")
        # Single-turn agent: only use the latest user question
        question = request.input[-1].content

        with dspy.context(lm=self.lm):
            output = self._streamified_agent(question=question)
            for chunk in output:
                converted_chunk = self._dspy_stream_chunk_to_responses(chunk)
                if isinstance(chunk, dspy.streaming.StatusMessage | dspy.Prediction):
                    yield ResponsesAgentStreamEvent(type="response.output_item.done", item=converted_chunk)
                elif isinstance(chunk, dspy.streaming.StreamResponse):
                    yield ResponsesAgentStreamEvent(**converted_chunk)
                    if chunk.is_last_chunk:
                        # The output field is finished, we yield the concatenated message with the same id.
                        text = "".join(self._concated_stream_chunks[-1])
                        self._concated_stream_chunks.append([])
                        yield ResponsesAgentStreamEvent(
                            type="response.output_item.done",
                            item=self.create_text_output_item(
                                text=text,
                                id=converted_chunk["item_id"],
                            ),
                        )

# Set model for logging or interactive testing
from mlflow.models import set_model
AGENT = DSPyResponsesAgent()
set_model(AGENT)

## 3. Test the agent
Interact with the agent to test its output. Because this notebook calls mlflow.dspy.autolog(), you can view the trace for each step the agent takes.

In [0]:
%restart_python

In [0]:
from agent import AGENT

input_example = {
    "input": [
        {
            "role": "user",
            "content": "Who is the top buyer in Australia?",
        }
    ]
}

output = AGENT.predict(input_example)
print(output.output[0].content[0]["text"])


In [0]:
from agent import AGENT

input_example = {
    "input": [
        {
            "role": "user",
            "content": "Who is the top buyer in USA?",
        }
    ]
}

output = AGENT.predict(input_example)
print(output.output[0].content[0]["text"])

## 4. Log the agent as an MLflow model
Log the agent as code from the `agent.py` file that was written by the previous cell. See [MLflow - Models from Code](https://mlflow.org/docs/latest/ml/model/#models-from-code).

### Enable automatic authentication for Databricks resources
For the most common Databricks resource types, Databricks supports and recommends declaring resource dependencies for the agent upfront during logging. This enables automatic authentication passthrough when you deploy the agent. With automatic authentication passthrough, Databricks automatically provisions, rotates, and manages short-lived credentials to securely access these resource dependencies from within the agent endpoint.

To enable automatic authentication, specify the dependent Databricks resources when calling mlflow.pyfunc.log_model().

In [0]:
import mlflow
from mlflow.models.resources import (
    DatabricksFunction,
    DatabricksGenieSpace,
    DatabricksServingEndpoint,
    DatabricksSQLWarehouse,
)
from pkg_resources import get_distribution

from mlflow.models import ModelConfig
config_file = "config.yaml"
model_config = mlflow.models.ModelConfig(development_config=config_file)

resources = [
    DatabricksServingEndpoint(endpoint_name= model_config.get("llm_endpoint_name")),
    DatabricksGenieSpace(genie_space_id= model_config.get("australian_sales_genie_space_id")),
    DatabricksGenieSpace(genie_space_id= model_config.get("us_retail_genie_space_id")),
    DatabricksSQLWarehouse(warehouse_id= model_config.get("sql_warehouse_id")),
]

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        model_config="config.yaml",
        extra_pip_requirements=[
            f"databricks-sdk=={get_distribution('databricks-sdk').version}",
            "dspy",
            "pydantic>=2.0.0,<3.0.0"
        ],
        resources=resources,
    )

### Pre-deployment agent validation
Before registering and deploying the agent, perform pre-deployment checks using the [mlflow.models.predict()](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.models.html#mlflow.models.predict) API. See Databricks documentation ([AWS](https://docs.databricks.com/aws/en/machine-learning/model-serving/model-serving-debug#validate-inputs) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/machine-learning/model-serving/model-serving-debug#before-model-deployment-validation-checks)).

In [0]:
mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/agent",
    input_data=input_example,
    env_manager="uv",
)

## 5. Register the Model in Unity Catalog

Update the `catalog`, `schema`, and `model_name` below to register the MLflow model to Unity Catalog.



In [0]:
mlflow.set_registry_uri("databricks-uc")

# TODO: define the catalog, schema, and model name for your UC model.
catalog = model_config.get("catalog")
schema = model_config.get("schema")
model_name = model_config.get("model_name")
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# register the model to UC
uc_registered_model_info = mlflow.register_model(model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME)

## 6. Deploy the agent

In [0]:
from databricks import agents
import os

agents.deploy(
  UC_MODEL_NAME, 
  uc_registered_model_info.version, 
  tags={"endpointSource": "docs"}
)

# Cleanup

In [0]:
# Delete the US retail tables
# delete_marketplace_tables(listing_id=simulated_retail_listing.id, catalog_suffix="us_retail")

# Delete the Australia sales tables
# delete_marketplace_tables(listing_id=australia_sales_listing.id, catalog_suffix="australia_sales")

## Next steps
After your agent is deployed, you can chat with it in AI playground to perform additional checks, share it with subject matter experts (SMEs) in your organization for feedback, or embed it in a production application. See Databricks documentation ([AWS](https://docs.databricks.com/en/generative-ai/deploy-agent.html) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/deploy-agent)).